# Setup

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os, re
import numpy as np
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, TwoSlopeNorm, LogNorm
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
import os, sys, configparser

settings_parser = configparser.ConfigParser()
settings_parser.read("localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
MIENC_PATH = settings_parser.get("global", "mienc_path")


band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
sns.set_theme("paper", "ticks")

cache_dir = os.path.join(MAIN_DATA_FOLDER, "cache")
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
FIGURES_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Figures")
if not os.path.isdir(FIGURES_FOLDER):
    os.mkdir(FIGURES_FOLDER)

sys.path.append(MIENC_PATH)
from mienc import Corrector

In [ ]:
def coloured_box_kwargs(colour):
    return {
        "boxprops": {"color": colour},
        "widths": 0.6,
        "whiskerprops": {"color": colour},
        "flierprops": {"markeredgecolor": colour, "marker": "+"},
        "medianprops": {"lw": 1.6, "color": colour},
        "capprops": {"color": colour},
    }

In [ ]:
def boxplot(
    x, series, colours, percents, ax=None, ttest=False, variant="Holm-Bonferroni"
):
    if ax is not None:
        plt.sca(ax)
    STRETCH = len(series)
    OFFSET = -STRETCH / 2
    handles = []
    labels = []

    for data, colour in zip(series, colours):
        plt.boxplot(
            series[data],
            positions=np.arange(len(x)) * STRETCH + OFFSET,
            **coloured_box_kwargs(colour),
        )
        handles.append(Patch(facecolor="white", edgecolor=colour))
        labels.append(data)
        OFFSET += 1

    for percent, style in zip(percents, ["solid", "dashed", "dashdot", "dotted"]):
        plt.hlines(
            percent / 100,
            -STRETCH,
            (len(x) - 0.5) * STRETCH,
            "r",
            style,
            lw=1,
            zorder=1,
        )
        handles.append(Line2D([0], [0], lw=1, color="r", linestyle=style))
        labels.append(f"{percent}%")

    if ttest:
        if hasattr(ttest, "len") and len(ttest) == 2:
            first, second = ttest
        elif len(series) == 2:
            first, second = series.keys()
        significance = np.array(
            [
                ttest_rel(a, b, alternative="greater").pvalue
                for a, b in zip(series[first], series[second])
            ]
        )
        if variant == "Bonferroni":
            threshold = 0.05 / len(x)
        else:
            moving_threshold = 0.05 / (len(x) - np.arange(len(x)))
            significance_sorter = np.argsort(significance)
            above_threshold = significance[significance_sorter] > moving_threshold
            if above_threshold.any():
                first_above = np.min(np.where(above_threshold))
                threshold = significance[significance_sorter[first_above]]
            else:
                threshold = 0.05

        markers = significance < threshold
        position = ((max(percents) / 100 if percents else 0) + plt.ylim()[1]) / 2
        ast = plt.scatter(
            np.arange(len(x))[markers] * STRETCH - OFFSET / 2,
            np.full(np.sum(markers), position, dtype=float),
            marker="*",
            color="k",
        )
        handles.append(ast)
        labels.append("Significative\ndifference")

    tickPos = np.arange(len(x)) * STRETCH
    plt.xticks(tickPos, x)
    plt.xlim((-STRETCH, (len(x) - 0.5) * STRETCH))
    plt.legend(handles, labels)
    return plt.gca()

# Amount of non-linearity
## fMRI

In [ ]:
regions = [
    10,
    30,
    50,
    70,
    100,
    150,
    200,
    230,
    270,
    300,
    350,
    400,
    450,
    500,
    550,
    600,
    650,
    700,
    750,
    800,
    850,
    900,
    950,
]
fMRI_results = {"True": [], "Shadow": []}
for region in regions:
    out_name = f"clean_cra_strin_{region}_bin7"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    fMRI_results["True"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    fMRI_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
fig = plt.figure(figsize=(12, 6))
boxplot(regions, fMRI_results, ["darkorange", "blue"], [0, 5], plt.gca(), True)
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.show()

## EEG

Data still missing

## iEEG

In [ ]:
sessions = [1, "long"]
iEEG_results = {session: {"True": [], "Shadow": []} for session in sessions}
for session in sessions:
    piece = session if session == "long" else f"ses{session:02}"
    band_folders = glob.glob(os.path.join(RESULTS_FOLDER, f"iEEG_{piece}_band*"))
    for folder in sorted(band_folders):
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        iEEG_results[session]["True"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
        iEEG_results[session]["Shadow"].append(
            1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"]
        )

In [ ]:
for session in [1, "long"]:
    fig = plt.figure(figsize=(12, 6))
    boxplot(
        band_names,
        iEEG_results[session],
        ["darkorange", "blue"],
        [0, 10],
        plt.gca(),
        True,
    )
    plt.xlabel("# regions")
    plt.ylabel("RNL")
    plt.title(f"Session: {session}")
    plt.show()

## Single unit spikes

In [ ]:
GOOD_SESSIONS = [
    831882777,
    816200189,
    771160300,
    786091066,
    779839471,
    778998620,
    781842082,
    778240327,
    793224716,
    839068429,
    794812542,
    768515987,
    767871931,
    840012044,
    766640955,
    847657808,
]
groups = {}
statsReg = [{} for __ in GOOD_SESSIONS]
for s, session in enumerate(GOOD_SESSIONS):
    for folder in glob.glob(os.path.join(RESULTS_FOLDER, f"spiking_{session}_*")):
        if os.path.isfile(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ):
            with open(os.path.join(folder, "shape.json")) as fp:
                groups[session] = json.load(fp)[1]
            timeWindow = int(re.findall(r"_(\d{4})_", folder)[0])
            with open(
                os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
            ) as fp:
                tmp = {k: np.array(v) for k, v in json.load(fp).items()}
            statsReg[s][timeWindow] = {
                "True": 1 - tmp["gaussMI"] / tmp["totalMI"],
                "Shadow": 1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"],
            }

In [ ]:
for data in ["True", "Shadow"]:
    spikes = np.empty((len(statsReg[0]), len(statsReg[0][125][data]), len(statsReg)))
    ylabels = list(sorted(statsReg[0].keys()))
    xlabels = [2**i for i in range(6)]
    for s in range(len(GOOD_SESSIONS)):
        for t, timeWindow in enumerate(sorted(statsReg[0].keys())):
            spikes[t, :, s] = statsReg[s][timeWindow][data]

    plt.imshow(np.mean(spikes, 2))
    plt.yticks(np.arange(7), ylabels)
    plt.xticks(np.arange(6), xlabels)
    cbar = plt.colorbar()
    cbar.ax.set_ylabel("RNL", rotation=90)
    cbar.ax.get_yaxis().labelpad = 15
    plt.xlabel("# units per site")
    plt.ylabel("Time sample [ms]")
    plt.title("Average")
    plt.show()

# Localisation of non-linearities
## fMRI

## EEG

# Sources of non-linearity
## fMRI

## EEG

## iEEG

### Seizure

In [ ]:
seizure = {}
for folder in glob.glob(os.path.join(RESULTS_FOLDER, "sampleSeizure_band?_bin*")):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        band = int(re.findall(r"band(\d+)", folder)[0])
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            tmp = {k: np.array(v) for k, v in json.load(fp).items()}
        seizure[band] = {
            "True": (tmp["totalMI"] - tmp["gaussMI"]) / tmp["totalMI"],
            "Shadow": (tmp["totalMIshadow"] - tmp["gaussMIshadow"])
            / tmp["totalMIshadow"],
        }
windowed_mean_power = np.load(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "sampleSeizure_power.npy")
)

In [ ]:
fs_fast = 200
RNL = []
bands = [[1.0, 4.0], [4.0, 8.0], [8.0, 12.0], [12.0, 30.0], [30.0, 44.0]]

sec_dest = 283.936
for i in range(1, 6):
    thisRNL = seizure[i]["True"]
    N_blocks = thisRNL.shape[0]
    fs_dest = int(bands[i - 1][1] * 2 * 1.125 + 0.5)
    slide_band = (fs_dest * 45) // 10
    fs_rnl = slide_band / fs_dest
    thisTime = fs_rnl * N_blocks
    tot_samp_dest = thisTime * fs_fast
    conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
    index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
    index[index == thisRNL.shape[0]] = thisRNL.shape[0] - 1
    RNL.append(thisRNL[index])
tmpRNL = np.concatenate(RNL)
normalised_wmp = (windowed_mean_power - np.min(windowed_mean_power)) / (
    np.max(windowed_mean_power) - np.min(windowed_mean_power)
) * (np.nanmax(tmpRNL) - np.nanmin(tmpRNL)) + np.nanmin(tmpRNL)
N_blocks = 64
fs_dest = 1024
slide_band = (fs_dest * 45) // 10
fs_rnl = slide_band / fs_dest
thisTime = fs_rnl * N_blocks
tot_samp_dest = thisTime * fs_fast
conversion_factor = tot_samp_dest / (fs_fast * fs_rnl)
index = np.round(np.arange(tot_samp_dest) / (fs_fast * fs_rnl)).astype(int)
index[index == normalised_wmp.shape[0]] = normalised_wmp.shape[0] - 1
RNL.append(normalised_wmp[index])

In [ ]:
tot_samp = max(map(lambda x: x.shape[0], RNL))
RNL_long = np.full([tot_samp, 6], np.nan)
for b in range(6):
    RNL_long[: RNL[b].shape[0], b] = RNL[b]

In [ ]:
plt.figure(figsize=(15, 8))
plt.imshow(RNL_long.T, interpolation="nearest")
plt.gca().set_aspect(5000)
# plt.show()
plt.xlabel("Time [s]")
plt.ylabel("Band")
plt.xticks(np.arange(12) * 5000, (np.arange(12) * 5000 / fs_fast).astype(int))
plt.yticks(
    np.arange(6),
    [r"$\propto$power", r"$\gamma$", r"$\beta$", r"$\alpha$", r"$\theta$", r"$\delta$"][
        ::-1
    ],
)
plt.ylim(plt.ylim())
plt.vlines(
    [50 * fs_fast, (183.936 + 50) * fs_fast],
    *plt.ylim(),
    colors="g",
    linestyles="solid",
    lw=2,
    zorder=20
)
for i in range(6):
    plt.vlines(
        [(50 - 45) * fs_fast, (183.936 + 50 - 45) * fs_fast],
        i - 0.5,
        i + 1 - 0.5,
        colors="b",
        linestyles="solid",
        lw=2,
        zorder=20,
    )
plt.colorbar(shrink=0.7).ax.set_ylabel("RNL", rotation=270, labelpad=15)
plt.xlim((0, 283.936 * fs_fast))
plt.show()

### Other Activity

In [ ]:
with open(
    os.path.join(MAIN_DATA_FOLDER, "iEEG", "spikes_by_subject_electrode.json")
) as fp:
    spikes_per_electrode = {
        int(k): np.array([len(v[a][0]) for a in v]) for k, v in json.load(fp).items()
    }
bands = [[1.0, 4.0], [4.0, 8.0], [8.0, 12.0], [12.0, 30.0], [30.0, 44.0]]
fs_dest = [int(band[1] * 2 * 1.125 + 0.5) for band in bands]

In [ ]:
def data_from(band, nbins, duration):
    MI_thres = {10: 0.015696, 13: 0.009054, 14: 0.008483, 20: 0.003671, 23: 0.002931}
    corrct = Corrector(
        nbins,
        duration,
        folder_name=os.path.join(RESULTS_FOLDER, f"iEEG_ses01_band{band}_bin{nbins}"),
        config="../config.ini",
        cache_dir="../cache/",
        display=False,
        verbose=True,
    )
    corrct.compute_correction()
    ns = []
    rnl = []
    aga = []
    for subj in range(18):
        MI = corrct.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"iEEG_ses01_band{band}_bin{nbins}",
                    f"subject{subj:02}_{nbins}.npy",
                )
            )
        )
        tr = MI[:, 0]
        ga = np.mean(MI[:, 1:], axis=1)
        rnl.append(1 - ga / tr)
        aga.append(ga)
        fe, se = np.triu_indices(24, 1)
        ns.append(
            spikes_per_electrode[subj + 1][fe]
            + spikes_per_electrode[subj + 1][se]  # [band-1][band-1]
        )
    ns = np.concatenate(ns)
    aga = np.concatenate(aga)
    rnl = np.concatenate(rnl)
    good = (rnl >= 0) & (rnl <= 1) & (aga > MI_thres[nbins])
    return ns, rnl, good

In [ ]:
data = [
    data_from(band, nbins, duration)
    for band, (duration, nbins) in enumerate(  # 23, 12276
        [(1116, 10), (2232, 13), (3348, 14), (8432, 20), (12276, 23)], start=1
    )
]
plt.close()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good, abt) in enumerate(data, start=1):
    good = np.isfinite(rnl) & (rnl > -5)
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good], rnl[good], norm=LogNorm()
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# spikes")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.show()
fig, ax = plt.subplots(2, 3, figsize=(15, 10), sharex=None, sharey="all")
plt.subplots_adjust(hspace=0.13, wspace=0)
for band, (ns, rnl, good, abt) in enumerate(data, start=1):
    his = ax[(band - 1) // 3, (band - 1) % 3].hist2d(
        ns[good], rnl[good], norm=LogNorm()
    )
    r = pearsonr(ns[good], rnl[good])
    ax[(band - 1) // 3, (band - 1) % 3].set_title(
        f"{band_names[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}",
        y=1.0,
        pad=5,
    )
    if (band - 1) // 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_xlabel("# spikes")
    if not (band - 1) % 3:
        ax[(band - 1) // 3, (band - 1) % 3].set_ylabel("RNL")
plt.colorbar(his[-1], ax=ax[1, 2]).set_label("# pairs")
ax[1, 2].axis("off")
plt.ylim(bottom=0, top=1)
plt.show()

## Single unit spikes

# Reliability of non-linearity
## fMRI test-retest

In [ ]:
corrector = Corrector(
    bins=8,
    duration=657,
    steps=200,
    iterations=50000,
    samples=657,
    folder_name=os.path.join(RESULTS_FOLDER, "lemon_aal_strin_90_ses-01_bin8"),
    cache_dir=cache_dir,
    verbose=True,
)
corrector.compute_correction()
all_sub = np.empty([3, 6, 14])
all_cov = np.empty([6, 6, 14])
for subject in range(14):
    all_series = np.empty([6, 4005])
    data = {}
    for session in range(3):
        all_series[2 * session, :] = corrector.correct(
            np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8.npy",
                )
            )[:, 0]
        )
        all_series[2 * session + 1, :] = -0.5 * np.log10(
            1
            - np.load(
                os.path.join(
                    RESULTS_FOLDER,
                    f"lemon_aal_strin_90_ses-0{session}_bin8/subject{subject:02}_8_cor.npy",
                )
            )
            ** 2
        )
    cov = np.corrcoef(all_series)
    np.fill_diagonal(cov, np.nan)
    all_cov[:, :, subject] = cov
    all_sub[:, :, subject] = np.diff(cov, axis=0)[::2, :]

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(np.mean(all_cov, axis=2))
plt.colorbar()
plt.yticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
)
plt.title("Average across 14 subjects")
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(np.mean(all_sub, axis=-1))
plt.colorbar(shrink=0.7).ax.set_ylabel(
    r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$",
    fontsize=14,
)
plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)], fontsize=15)
plt.xticks(
    range(6),
    labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    rotation=90,
    fontsize=15,
)
plt.ylabel("Predictors from:", fontsize=16)
plt.xlabel("Predicting:", fontsize=16)
plt.title("Average across 14 subjects", fontsize=18)
plt.show()

## EEG

## iEEG

In [ ]:
all_sub = {band: np.empty([3, 6, 18]) for band in range(1, 6)}
all_cov = {band: np.empty([6, 6, 18]) for band in range(1, 6)}
for band in range(1, 6):
    for subject in range(18):
        all_series = np.empty([6, 276])
        data = {}
        for session in range(3):
            band_folder = glob.glob(
                os.path.join(RESULTS_FOLDER, f"iEEG_ses{session+1:02}_band{band}_bin*")
            )[0]
            bins = int(band_folder.split("bin")[1])
            with open(os.path.join(band_folder, "shape.json"), "r") as fp:
                duration = json.load(fp)[0]
            corrector = Corrector(
                bins=bins,
                duration=duration,
                steps=200,
                iterations=50000,
                samples=duration,
                folder_name=band_folder,
                cache_dir=cache_dir,
                verbose=False,
            )
            corrector.compute_correction()
            all_series[2 * session, :] = corrector.correct(
                np.load(os.path.join(band_folder, f"subject{subject:02}_{bins}.npy"))[
                    :, 0
                ]
            )
            all_series[2 * session + 1, :] = -0.5 * np.log10(
                1
                - np.load(
                    os.path.join(band_folder, f"subject{subject:02}_{bins}_cor.npy")
                )
                ** 2
            )
        cov = np.corrcoef(all_series)
        np.fill_diagonal(cov, np.nan)
        all_cov[band][:, :, subject] = cov
        all_sub[band][:, :, subject] = np.diff(cov, axis=0)[::2, :]

In [ ]:
for band in all_cov:
    plt.figure(figsize=(10, 10))
    plt.imshow(np.mean(all_cov[band], axis=2))
    plt.colorbar()
    plt.yticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
    )
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
    )
    plt.title(f"Average across 18 subjects $-$ Band {band_names[band-1]}")
    plt.show()

In [ ]:
for band in all_sub:
    mean_all_sub = np.mean(all_sub[band], axis=-1)
    max_abs = np.nanmax(np.abs(mean_all_sub))
    norm = TwoSlopeNorm(vcenter=0, vmax=max_abs, vmin=-max_abs)
    plt.figure(figsize=(15, 10))
    plt.imshow(mean_all_sub, cmap=plt.cm.RdYlBu_r, norm=norm)
    plt.colorbar(shrink=0.7).ax.set_ylabel(
        r"Corr$\langle -0.5*\log_{10}(1-\rho^2) \rangle-$Corr$\langle TMI \rangle$",
        fontsize=14,
    )
    plt.yticks(range(3), labels=[f"{i+1} ses" for i in range(3)], fontsize=15)
    plt.xticks(
        range(6),
        labels=[f"{i+1} ses - {j}" for i in range(3) for j in ["TMI", r"$I_{COR}$"]],
        rotation=90,
        fontsize=15,
    )
    plt.ylabel("Predictors from:", fontsize=16)
    plt.xlabel("Predicting:", fontsize=16)
    plt.title(f"Average across 18 subjects $-$ Band {band_names[band-1]}", fontsize=18)
    plt.show()